# Lab 2 - QC-AFQMC with Amazon Braket and AWS Batch

## Introduction

In this lab, we will run a hybrid quantum-classical AFQMC workload for the hydrogen molecule using both Amazon Braket and AWS Batch. We use the [Amazon Braket Hybrid Job](https://docs.aws.amazon.com/braket/latest/developerguide/braket-what-is-hybrid-job.html) feature to perform shadow tomography. Then AWS Batch helps you to run the successive post-processing workloads using classical computing resources, in a similar way to our previous lab. 

In the following, we will go through the steps to
1. Create a hybrid job for shadow tomography using 4000 circuits (100 shots each)
2. Run a large scale QC-AFQMC example with 200 child jobs running on multiple compute instances in parallel
3. Compare the results of QC-AFQMC calculation with the previous classical AFQMC calculation

Let's create the boto3 clients for the services we will use in this lab. Some additional packages are needed for this section, so we'll install them first.

In [ ]:
import boto3

sts_client = boto3.client("sts")
s3_client = boto3.client("s3")
cfn_client = boto3.client("cloudformation")
batch_client = boto3.client("batch")

ACCOUNT_ID = sts_client.get_caller_identity().get("Account")
print("Account ID:", ACCOUNT_ID)

my_session = boto3.session.Session()
WORKING_REGION = my_session.region_name
print("Region:", WORKING_REGION)

In [ ]:
for output in cfn_client.describe_stacks(StackName='batch-environment').get('Stacks')[0].get('Outputs'):
    if output.get('OutputKey') == "DataBucketName":
        print(f"{output.get('OutputKey')}: {output.get('OutputValue')}")
        data_bucket_name = output.get('OutputValue')

## Create a hybrid job

![](images/braket-hybrid-job.png)

Amazon Braket Hybrid Job offers a way for you to run hybrid quantum-classical algorithms requiring both classical AWS resources and quantum processing units (QPUs). Hybrid Jobs is ideal for long-running, iterative algorithms that involve the use of both classical computing resources and quantum computing resources. With Hybrid Jobs, after submitting your algorithm to run, Braket will run your algorithm in a scalable, containerized environment. Once the algorithm has completed, you can then retrieve the results.

Additionally, quantum tasks that are created from a hybrid job benefit from higher priority queueing to the target QPU device. This prioritization ensures that your quantum computations are processed and ran ahead of other tasks waiting in the queue. This is particularly advantageous for iterative hybrid algorithms, where the results of one quantum task depend on the outcomes of prior quantum tasks.

Specifically for our job, we will specify the following
- `device`: on which hardware the job is to be run
- `source_module`: folder of source code
- `entry_point`: which function is to be executed by the job
- `job_name`: job name
- `instance_config`: which type of EC2 instance is the classical part of the job hosted
- `output_data_config`: S3 bucket for storing the output of this job
- `hyperparameters`
  - `ShadowSize`: number of shadow circuits
  - `Shots`: number of shots for each circuit
  
Specifically, we include a [`requirements.txt`](./afqmc/requirements.txt) file inside the source folder, and the packages will be installed on the fly at the beginning of the job. In addition, you can also provide custom container image for your job.

### Provide the Braket hybrid jobs code

Let's package and upload the code for the hybrid job to generate matchgate shadows with quantum computing resources to our S3 bucket.

In [ ]:
import tarfile
import tempfile
from pathlib import Path

afqmc_path = Path("./afqmc").resolve(strict=True)
entrypoint = Path("./collect_shadows.py").resolve(strict=True)

with tempfile.TemporaryDirectory() as temp_dir:
    filename = "afqmc.tar.gz"
    filepath = f"{temp_dir}/{filename}"
    print(f"Tarring up source code: {filepath}")
    with tarfile.open(filepath, "w:gz", dereference=True) as tar:
        tar.add(afqmc_path, arcname=afqmc_path.name)
        tar.add(entrypoint, arcname=f"{afqmc_path.name}/{entrypoint.name}")
    print(f"Uploading tarball to S3: {data_bucket_name}/{filename}") 
    s3_client.upload_file(filepath, data_bucket_name, filename)
    s3_uri = f"s3://{data_bucket_name}/{filename}"
    print(f"The S3 URI of the uploaded tarball: {s3_uri}")

### Run the hybrid job

In [ ]:
from braket.aws import AwsQuantumJob
from braket.jobs.config import InstanceConfig, OutputDataConfig
from braket.devices import Devices
import time

timestamp = str(int(time.time() * 1000))
hybrid_job_output_prefix = f"braket/lab-2-qc-afqmc-matchgate-circuits/{timestamp}"

stack = cfn_client.describe_stacks(StackName="AFQMC-labs-infrastructure")["Stacks"][0]
outputs = {o["OutputKey"]: o["OutputValue"] for o in stack["Outputs"]}
hybrid_job_role_arn = outputs["BraketJobsExecutionRoleArn"]


job = AwsQuantumJob.create(
    device=Devices.Amazon.SV1,
    source_module=s3_uri,
    entry_point="afqmc.collect_shadows:main",
    job_name="lab-2-afqmc-matchgate-circuits",
    instance_config=InstanceConfig(instanceType="ml.m5.xlarge"),
    output_data_config=OutputDataConfig(
        s3Path=f"s3://{data_bucket_name}/{hybrid_job_output_prefix}",
    ),
    hyperparameters={
        "ShadowSize": "4000",
        "Shots": "100",
    },
    role_arn=hybrid_job_role_arn
)

In [ ]:
job.state()

Once the hybrid job starts, it will change the status to `RUNNING`. We can also check the hybrid job status in the Braket console. The quantum tasks in this tutorial are executed on the Amazon Braket SV1 managed state vector simulator which can execute up to 100 tasks concurrently in your account. 

<div class="alert alert-block alert-success">
<b>Activity:</b> Navigate to the <a href="https://us-east-1.console.aws.amazon.com/braket/home?region=us-east-1#/tasks">Amazon Braket management console</a> to check the status of your quantum tasks (see the screenshot below).
</div>

![](./images/braket-tasks.png)

<div class="alert alert-block alert-success">
<b>Activity:</b> Navigate to the <a href="https://us-east-1.console.aws.amazon.com/cloudwatch/home?region=us-east-1#metricsV2?graph=~(metrics~(~(~'AWS*2fBraket~'CREATED~'Type~'Tasks~'DeviceArn~'arn*3aaws*3abraket*3a*3a*3adevice*2fquantum-simulator*2famazon*2fsv1~(id~'m1))~(~'.~'COMPLETED~'.~'.~'.~'.~(id~'m2)))~view~'timeSeries~stacked~false~region~'us-east-1~stat~'Sum~period~10~start~'-PT10M~end~'P0D)&query=~'*7bAWS*2fBraket*2cDeviceArn*2cType*7d">Amazon CloudWatch console</a> to monitor task metrics in near-real time (see the screenshot below).
</div>

![](./images/cloudwatch-metrics.png)

<div class="alert alert-block alert-success">
<b>Activity:</b> Once your job is in the COMPLETED state, navigate to the S3 management console and to locate the results file.
</div>

In [ ]:
measurement_output = job.result()['output']
Q_save = job.result()['Q_save']

## Large scale post-processing with Batch

So far, we have used Braket Hybrid Job to run matchgate circuits and collect measurement outcomes. Now we proceed to the classical post-processing step. We start with updaing the Batch job definition, since the post-processing of classical shadows requires more memory.

### Update job definition

<div class="alert alert-block alert-success">
<b>Activity:</b> Update the CloudFormation stack to increase the resource memory in the job definition to <b>2048</b>.
</div>

In [ ]:
with open('batch-environment.yaml', 'r') as file:
    template_body = file.read()

stack_name = 'batch-environment'
try:
    cfn_client.update_stack(
        StackName=stack_name,
        TemplateBody=template_body,
        Parameters=[
            {'ParameterKey': 'JobDefResourceMemory', 'ParameterValue': '2048'}
        ]
    )
    print("Waiting for CloudFormation stack to be updated...")
    waiter = cfn_client.get_waiter("stack_update_complete")
    waiter.wait(
        StackName=stack_name,
        WaiterConfig={'Delay': 10, 'MaxAttempts': 150}
    )
    print("CloudFormation stack updated.")
except cfn_client.exceptions.ClientError as e:
    print(e)

print()
for output in cfn_client.describe_stacks(StackName=stack_name).get('Stacks')[0].get('Outputs'):
    print(f"{output.get('OutputKey')}: {output.get('OutputValue')}")
    if output.get('OutputKey') == "DataBucketName":
        data_bucket_name = output.get('OutputValue')
    if output.get('OutputKey') == "JobQueue":
        job_queue = output.get('OutputValue')
    if output.get('OutputKey') == "JobDefinition":
        job_definition = output.get('OutputValue')

<div class="alert alert-block alert-success">
<b>Activity:</b> Navigate to the <a href="https://us-east-1.console.aws.amazon.com/cloudformation/home">AWS CloudFormation mangement console</a> again to check the status of your stack and see the individual resources being created (see the screenshot below).
</div>

### Run the Batch job

Next we submit the Batch job, just like we did in lab-1. Note that we have to specify one additional environment variable `JOB_INPUT_FILE_KEY`, pointing to the S3 location where the classical shadows are saved.

In [ ]:
response = batch_client.submit_job(
    jobName="lab-2-quantum-large-scale",
    jobQueue=job_queue,
    jobDefinition=job_definition,
    parameters={
        "entry_point": "qc_afqmc",
        "delta_tau": "0.005",
        "time_steps": "200",
        "input_file_key": f"{hybrid_job_output_prefix}/output/model.tar.gz"
    },
    arrayProperties={"size": 400}
)

job_id = response["jobId"]

<div class="alert alert-block alert-success">
<b>Activity:</b> Navigate to the <a href="https://us-east-1.console.aws.amazon.com/batch/home">AWS Batch mangement console</a> and find the job you submitted above.
</div>

In [ ]:
batch_client.describe_jobs(jobs=[job_id]).get("jobs")[0].get("status")

## Retrieve results of large-scale post-processing job

Finally, let's summarize the QC-AFQMC calculation results, and compare it against its classical counterpart.

In [ ]:
import json
import numpy as np
from pathlib import Path

local_energies_real = []
local_energies_imag = []
weights = []

for i in range(400):
    with open(f"results/lab-1/result_{i}.json", "r") as file:
        data = json.load(file)
    [local_energies_real.append(j) for j in data["local_energies_real"]]
    [local_energies_imag.append(j) for j in data["local_energies_imag"]]
    [weights.append(j) for j in data["weights"]]

local_energies = [[ii+1.j*jj for ii, jj in zip(i, j)] for i, j in zip(local_energies_real, local_energies_imag)]   
classical_energies = np.real(np.average(local_energies, weights=weights, axis=0))

In [ ]:
import json
import numpy as np
from pathlib import Path

job_status = batch_client.describe_jobs(jobs=[job_id]).get("jobs")[0].get("status")
if job_status == "SUCCEEDED":
    Path("results/lab-2").mkdir(parents=True, exist_ok=True)
    s3_client = boto3.client("s3")
    for i in range(400):
        s3_client.download_file(
            data_bucket_name,
            f"batch/{job_id}/{i}/results.json",
            f"results/lab-2/result_{i}.json"
        )
else:
    print(f"Your job is in status {job_status}")

local_energies_real = []
local_energies_imag = []
weights = []

for i in range(400):
    with open(f"results/lab-2/result_{i}.json", "r") as file:
        data = json.load(file)
    [local_energies_real.append(j) for j in data["local_energies_real"]]
    [local_energies_imag.append(j) for j in data["local_energies_imag"]]
    [weights.append(j) for j in data["weights"]]

local_energies = [[ii+1.j*jj for ii, jj in zip(i, j)] for i, j in zip(local_energies_real, local_energies_imag)]   
qc_energies = np.real(np.average(local_energies, weights=weights, axis=0))

In [ ]:
import matplotlib.pyplot as plt

%matplotlib inline

plt.plot(
    0.005 * np.arange(600),
    classical_energies,
    linestyle="dashed",
    marker=".",
    color="tab:blue",
    label="classical",
)
plt.plot(
    0.005 * np.arange(200),
    qc_energies,
    linestyle="dashed",
    marker=".",
    color="tab:orange",
    label="quantum",
)
plt.axhline(-1.137117067345732, linestyle="dashed", color="black", label="reference")
plt.title(r"Ground state estimation of H$_2$ using AFQMC", fontsize=16)
plt.legend(fontsize=14, loc="upper right")
plt.xlabel(r"$\tau$ $(Hartree^{-1})$", fontsize=14)
plt.ylabel("Energy $(Hartree)$", fontsize=14)
plt.yticks(fontsize=14)
plt.tick_params(direction="in", labelsize=14)
plt.show()

## Wrapping up

<div class="alert alert-block alert-info">
<b>You reached the end of lab 2. Well done!</b>
</div>